# SPINN GSoC 2026 — PINN Test Submission
## Solving the 2D Poisson Equation with Physics-Informed Neural Networks

**Author:** Samarjeet Malik  
**Affiliation:** KIIT Bhubaneswar (B.Tech CS) | Pre-Doctoral Fellow, IIT Jodhpur  
**Date:** March 2026

---

### Problem Statement

We solve the 2D Poisson equation on the unit square Ω = [0,1]²:

$$\nabla^2 u(x,y) = f(x,y)$$

with homogeneous Dirichlet boundary conditions $u = 0$ on $\partial\Omega$.

**Why this problem?** The SPINN project focuses on magnetostatic shape optimization where the governing PDE is $\nabla^2 A = -\mu J$ (Poisson equation for the magnetic vector potential). This test directly demonstrates proficiency with the core PDE class used in the project.

**Analytical solution:** $u(x,y) = \sin(\pi x) \cdot \sin(\pi y)$  
**Forcing term:** $f(x,y) = -2\pi^2 \sin(\pi x) \cdot \sin(\pi y)$

### Approach

1. **Architecture:** Fully-connected network with tanh activation (matching Zhang et al., 2024)
2. **Loss function:** $\mathcal{L} = \lambda_{pde} \cdot \mathcal{L}_{pde} + \lambda_{bc} \cdot \mathcal{L}_{bc}$
3. **Training:** Adam optimizer with cosine annealing LR schedule
4. **Validation:** Quantitative comparison against analytical solution

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

# Use GPU if available (Colab T4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. PINN Architecture

We use a fully-connected neural network: **Input(2) → [50 neurons × 6 hidden layers, tanh] → Output(1)**

Key design choices:
- **tanh activation:** Provides infinitely differentiable outputs, essential for accurate Laplacian computation via autograd. This matches the architecture in Zhang et al. (2024).
- **Xavier initialization:** Prevents vanishing/exploding gradients in deep tanh networks.
- **6 × 50 architecture:** Directly matches the SPINN project reference paper.

In [ ]:
class PINN(nn.Module):
    """
    Physics-Informed Neural Network for solving the 2D Poisson equation.
    Architecture matches Zhang et al. (2024, Sci. Reports) — the SPINN project paper.
    """
    def __init__(self, hidden_layers=6, neurons_per_layer=50):
        super(PINN, self).__init__()

        layers = []
        # Input layer: (x, y) → hidden
        layers.append(nn.Linear(2, neurons_per_layer))
        layers.append(nn.Tanh())

        # Hidden layers
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(neurons_per_layer, neurons_per_layer))
            layers.append(nn.Tanh())

        # Output layer: hidden → u(x,y)
        layers.append(nn.Linear(neurons_per_layer, 1))

        self.network = nn.Sequential(*layers)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.network:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)

# Instantiate
model = PINN(hidden_layers=6, neurons_per_layer=50).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Architecture: 2 → [50 × 6 hidden, tanh] → 1")

## 2. Analytical Solution and Forcing Term

For validation, we choose $u(x,y) = \sin(\pi x) \sin(\pi y)$, which satisfies:
- $u = 0$ on all boundaries of $[0,1]^2$ ✓
- $\nabla^2 u = -2\pi^2 \sin(\pi x)\sin(\pi y) = f(x,y)$ ✓

In [ ]:
def analytical_solution(x, y):
    """Exact solution: u(x,y) = sin(πx) · sin(πy)"""
    return torch.sin(np.pi * x) * torch.sin(np.pi * y)

def forcing_term(x, y):
    """
    Source term derived from the analytical solution.
    ∇²u = ∂²u/∂x² + ∂²u/∂y² = -π²sin(πx)sin(πy) - π²sin(πx)sin(πy)
         = -2π² sin(πx) sin(πy)
    """
    return -2.0 * (np.pi ** 2) * torch.sin(np.pi * x) * torch.sin(np.pi * y)

## 3. Collocation Point Sampling

We use **stochastic collocation**: points are re-sampled every epoch. This provides:
- Implicit regularization (prevents overfitting to fixed point locations)
- Better domain coverage over training
- Analogous to mini-batch SGD for data-driven training

In [ ]:
def sample_interior_points(n_points):
    """Random collocation points in the interior of [0,1]² for PDE enforcement."""
    x = torch.rand(n_points, 1, device=device, requires_grad=True)
    y = torch.rand(n_points, 1, device=device, requires_grad=True)
    return x, y

def sample_boundary_points(n_per_edge):
    """
    Sample points on all four edges of the unit square for BC enforcement.
    Separate sampling per edge ensures uniform boundary coverage.
    """
    # Bottom (y=0), Top (y=1), Left (x=0), Right (x=1)
    x_b = torch.rand(n_per_edge, 1, device=device)
    y_b = torch.zeros(n_per_edge, 1, device=device)

    x_t = torch.rand(n_per_edge, 1, device=device)
    y_t = torch.ones(n_per_edge, 1, device=device)

    x_l = torch.zeros(n_per_edge, 1, device=device)
    y_l = torch.rand(n_per_edge, 1, device=device)

    x_r = torch.ones(n_per_edge, 1, device=device)
    y_r = torch.rand(n_per_edge, 1, device=device)

    x_bc = torch.cat([x_b, x_t, x_l, x_r], dim=0)
    y_bc = torch.cat([y_b, y_t, y_l, y_r], dim=0)
    return x_bc, y_bc

## 4. PDE Residual via Automatic Differentiation

The core of the PINN: we compute $\nabla^2 u_{pred} - f(x,y)$ using PyTorch autograd.

This is the **same mechanism** used in the SPINN project for computing the magnetostatic PDE residual $\nabla^2 A + \mu J = 0$ (Raissi et al., 2019).

In [ ]:
def compute_pde_residual(model, x, y):
    """
    Compute PDE residual: r = ∇²u_pred - f(x,y)

    Uses PyTorch autograd for second derivatives:
        ∇²u = ∂²u/∂x² + ∂²u/∂y²

    The residual should → 0 as the PINN learns to satisfy the PDE.
    """
    xy = torch.cat([x, y], dim=1)
    u = model(xy)

    # First derivatives via autograd
    grads = torch.autograd.grad(u, [x, y],
                                grad_outputs=torch.ones_like(u),
                                create_graph=True)
    u_x, u_y = grads[0], grads[1]

    # Second derivatives (Laplacian components)
    u_xx = torch.autograd.grad(u_x, x,
                               grad_outputs=torch.ones_like(u_x),
                               create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y,
                               grad_outputs=torch.ones_like(u_y),
                               create_graph=True)[0]

    laplacian = u_xx + u_yy
    f = forcing_term(x, y)

    # Residual: should be zero when PDE is satisfied
    return laplacian - f

## 5. Training

$$\mathcal{L} = \mathcal{L}_{pde} + \lambda_{bc} \cdot \mathcal{L}_{bc}$$

where:
- $\mathcal{L}_{pde} = \frac{1}{N_{int}} \sum_i (\nabla^2 u_{pred} - f)^2$ — PDE residual at interior points
- $\mathcal{L}_{bc} = \frac{1}{N_{bc}} \sum_j u_{pred}^2$ — Boundary condition residual
- $\lambda_{bc} = 10$ — Stronger weight on BCs since exact satisfaction is critical

**Loss weighting rationale:** Under-weighting BC loss leads to solutions that approximately satisfy the PDE but violate boundary conditions — a well-documented PINN failure mode (Cuomo et al., 2022, §4.2).

In [ ]:
def train_pinn(model, n_epochs=15000, n_interior=4000, n_boundary=500,
               lr=1e-3, lambda_bc=10.0):
    """
    Train the PINN with Adam + cosine annealing LR schedule.

    Hyperparameters:
    - n_interior=4000: collocation points per epoch (stochastically sampled)
    - n_boundary=500: boundary points per edge per epoch
    - lambda_bc=10: BC enforcement weight
    - lr=1e-3: initial learning rate with cosine decay
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = {
        'total_loss': [], 'pde_loss': [], 'bc_loss': [],
        'l2_error': [], 'epoch': []
    }

    print(f"{'Epoch':>6} | {'Total Loss':>12} | {'PDE Loss':>12} | "
          f"{'BC Loss':>12} | {'L2 Rel Err':>12}")
    print("-" * 72)

    start_time = time.time()

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()

        # ---- PDE residual loss (interior) ----
        x_int, y_int = sample_interior_points(n_interior)
        residual = compute_pde_residual(model, x_int, y_int)
        loss_pde = torch.mean(residual ** 2)

        # ---- Boundary condition loss ----
        x_bc, y_bc = sample_boundary_points(n_boundary)
        xy_bc = torch.cat([x_bc, y_bc], dim=1)
        u_bc_pred = model(xy_bc)
        loss_bc = torch.mean(u_bc_pred ** 2)  # Dirichlet: u=0

        # ---- Total weighted loss ----
        loss = loss_pde + lambda_bc * loss_bc
        loss.backward()
        optimizer.step()
        scheduler.step()

        # ---- Log every 500 epochs ----
        if epoch % 500 == 0 or epoch == n_epochs - 1:
            model.eval()
            with torch.no_grad():
                xx = torch.linspace(0, 1, 100, device=device)
                yy = torch.linspace(0, 1, 100, device=device)
                X, Y = torch.meshgrid(xx, yy, indexing='ij')
                xy_eval = torch.stack([X.flatten(), Y.flatten()], dim=1)
                u_pred = model(xy_eval).reshape(100, 100)
                u_exact = analytical_solution(X, Y)
                l2_err = torch.norm(u_pred - u_exact) / torch.norm(u_exact)

            history['total_loss'].append(loss.item())
            history['pde_loss'].append(loss_pde.item())
            history['bc_loss'].append(loss_bc.item())
            history['l2_error'].append(l2_err.item())
            history['epoch'].append(epoch)

            print(f"{epoch:6d} | {loss.item():12.6e} | {loss_pde.item():12.6e} | "
                  f"{loss_bc.item():12.6e} | {l2_err.item():12.6e}")

    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed:.1f} seconds")
    print(f"Final L2 relative error: {history['l2_error'][-1]:.6e}")
    return history

# Run training
history = train_pinn(model)

## 6. Results Visualization

In [ ]:
# Evaluate on fine grid
model.eval()
n_plot = 200
xx = torch.linspace(0, 1, n_plot, device=device)
yy = torch.linspace(0, 1, n_plot, device=device)
X, Y = torch.meshgrid(xx, yy, indexing='ij')

with torch.no_grad():
    xy_eval = torch.stack([X.flatten(), Y.flatten()], dim=1)
    U_pred = model(xy_eval).reshape(n_plot, n_plot).cpu().numpy()
    U_exact = analytical_solution(X, Y).cpu().numpy()
    X_np, Y_np = X.cpu().numpy(), Y.cpu().numpy()

error = np.abs(U_pred - U_exact)

# ---- Figure 1: Solution Comparison ----
fig = plt.figure(figsize=(18, 5))
gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 0.05])

vmin, vmax = 0, 1.0

ax1 = fig.add_subplot(gs[0])
ax1.contourf(X_np, Y_np, U_pred, levels=50, cmap='viridis', vmin=vmin, vmax=vmax)
ax1.set_title('PINN Prediction', fontsize=14, fontweight='bold')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_aspect('equal')

ax2 = fig.add_subplot(gs[1])
ax2.contourf(X_np, Y_np, U_exact, levels=50, cmap='viridis', vmin=vmin, vmax=vmax)
ax2.set_title('Analytical Solution', fontsize=14, fontweight='bold')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_aspect('equal')

ax3 = fig.add_subplot(gs[2])
c3 = ax3.contourf(X_np, Y_np, error, levels=50, cmap='hot_r')
ax3.set_title('Pointwise Absolute Error', fontsize=14, fontweight='bold')
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_aspect('equal')
plt.colorbar(c3, cax=fig.add_subplot(gs[3]))

plt.tight_layout()
plt.show()

print(f"\nMax absolute error: {error.max():.6e}")
print(f"Mean absolute error: {error.mean():.6e}")
print(f"PINN max value: {U_pred.max():.6f}  (exact: {U_exact.max():.6f})")

In [ ]:
# ---- Figure 2: Loss Curves and Cross-Section ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = history['epoch']

axes[0].semilogy(epochs, history['total_loss'], 'b-', lw=1.5, label='Total')
axes[0].semilogy(epochs, history['pde_loss'], 'r--', lw=1.5, label='PDE')
axes[0].semilogy(epochs, history['bc_loss'], 'g-.', lw=1.5, label='BC')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title('Training Loss Evolution', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(epochs, history['l2_error'], 'k-', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('L2 Relative Error (log)')
axes[1].set_title('Convergence', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Cross-section at y = 0.5
x_cs = torch.linspace(0, 1, 200, device=device)
y_cs = 0.5 * torch.ones(200, device=device)
with torch.no_grad():
    xy_cs = torch.stack([x_cs, y_cs], dim=1)
    u_cs_pred = model(xy_cs).cpu().numpy().flatten()
    u_cs_exact = analytical_solution(x_cs, y_cs).cpu().numpy()

axes[2].plot(x_cs.cpu().numpy(), u_cs_exact, 'b-', lw=2.5, label='Analytical')
axes[2].plot(x_cs.cpu().numpy(), u_cs_pred, 'r--', lw=2, label='PINN')
axes[2].set_xlabel('x'); axes[2].set_ylabel('u(x, 0.5)')
axes[2].set_title('Cross-section at y = 0.5', fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Figure 3: 3D Surface Comparison ----
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X_np, Y_np, U_pred, cmap='viridis', alpha=0.9)
ax1.set_title('PINN Prediction', fontweight='bold')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('u')

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X_np, Y_np, U_exact, cmap='viridis', alpha=0.9)
ax2.set_title('Analytical Solution', fontweight='bold')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('u')

plt.tight_layout()
plt.show()

## 7. Convergence Analysis: Effect of Collocation Point Density

We investigate how the number of interior collocation points affects solution accuracy, keeping all other hyperparameters fixed. This demonstrates understanding of a key convergence factor in PINNs.

In [ ]:
print("Effect of collocation point density (5000 epochs each):\n")

results = []
for n_pts in [500, 1000, 2000, 4000]:
    torch.manual_seed(42)
    test_model = PINN(hidden_layers=4, neurons_per_layer=40).to(device)
    opt = torch.optim.Adam(test_model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5000)

    for epoch in range(5000):
        opt.zero_grad()
        x_i, y_i = sample_interior_points(n_pts)
        res = compute_pde_residual(test_model, x_i, y_i)
        l_pde = torch.mean(res ** 2)

        x_b, y_b = sample_boundary_points(200)
        l_bc = torch.mean(test_model(torch.cat([x_b, y_b], dim=1)) ** 2)

        (l_pde + 10.0 * l_bc).backward()
        opt.step(); sched.step()

    test_model.eval()
    with torch.no_grad():
        xx = torch.linspace(0, 1, 100, device=device)
        yy = torch.linspace(0, 1, 100, device=device)
        Xg, Yg = torch.meshgrid(xx, yy, indexing='ij')
        u_p = test_model(torch.stack([Xg.flatten(), Yg.flatten()], dim=1)).reshape(100,100)
        u_e = analytical_solution(Xg, Yg)
        err = (torch.norm(u_p - u_e) / torch.norm(u_e)).item()

    results.append((n_pts, err))
    print(f"  N_interior = {n_pts:5d}  →  L2 relative error = {err:.6e}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ns, errs = zip(*results)
ax.loglog(ns, errs, 'bo-', markersize=8, linewidth=2)
ax.set_xlabel('Number of Interior Collocation Points', fontsize=12)
ax.set_ylabel('L2 Relative Error', fontsize=12)
ax.set_title('Convergence vs Collocation Density', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n→ Denser collocation sampling improves PDE residual coverage,")
print("  leading to more accurate solutions.")

## 8. Summary and Connection to SPINN Project

### What we demonstrated
- Built a PINN from scratch in PyTorch to solve the 2D Poisson equation $\nabla^2 u = f$
- Achieved low L2 relative error against the known analytical solution
- Showed clear convergence through loss curves and error evolution
- Analyzed the effect of collocation point density on accuracy

### Direct connection to SPINN

The SPINN project (Zhang et al., 2024) couples two neural networks:

| Network | Role | Equation |
|---------|------|----------|
| $NN_\theta$ (Physics) | Predicts magnetic vector potential $A(x,y)$ | $\nabla^2 A = -\mu J$ (Poisson!) |
| $NN_\phi$ (Shape) | Projects reference → optimized coordinates | Coordinate projection |

**The PINN demonstrated here is the foundation of $NN_\theta$.** The same architecture (6×50, tanh), loss formulation (PDE residual + boundary conditions), and training approach (Adam + LR decay) are used in the reference paper.

### Skills demonstrated
- PyTorch autograd for PDE residuals (2nd-order automatic differentiation)
- Multi-objective loss balancing (PDE + BC with weighting)
- Stochastic collocation point sampling
- Quantitative validation against analytical solutions
- Convergence analysis and understanding of PINN failure modes

### References
1. Zhang, Z., Lin, C., & Wang, B. (2024). Physics-informed shape optimization using coordinate projection. *Scientific Reports*, 14, 6537.
2. Cuomo, S. et al. (2022). Scientific Machine Learning through Physics-Informed Neural Networks. *J. Sci. Comput.*, 92, 88.
3. Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). Physics-informed neural networks. *J. Comput. Phys.*, 378, 686–707.